# How long does an RNN remember a fact you told it?

This notebook measures one thing, carefully:

> Give an RNN language model a fact, bury it under a growing pile of noise, then ask about it.
> **At what point does it stop being able to answer?**

Every probe has the same three-part shape:

```
FACT   ->   NOISE   ->   QUESTION
"My name is Alderic."   ...0 to 25,000 words...   "Question: What is my name?  Answer:"
```

The model is **RWKV-7 "Goose" World 1.5B** (`RWKV-x070-World-1.5B-v3`). RWKV is a recurrent
network, not a transformer: instead of a KV-cache that grows with every token, it carries a
**fixed-size state** — 24 layers x (a 2048-vector, a 32x64x64 matrix, another 2048-vector), about
12 MB, and it is *exactly the same size* after 25,000 words as after five. There is no attention
window to fall out of. The fact either survives in that state or it is overwritten.

That also means 25,000 words is a fair thing to ask for. It is roughly **34,000 tokens, about 8x
the checkpoint's 4,096-token training context** — a transformer of this size simply could not be
fed it, but an RNN can. Whether it *remembers* anything is the experiment.

## What is measured

Four conditions, 20 distinct fact items each, 13 noise budgets — **1,040 probes**.

| condition | fact given? | what the noise is | what it tests |
|---|---|---|---|
| `control` | **no** | Wikipedia text | The chance floor. If the model "recalls" a name it was never told, every other number here is inflated. |
| `repeat` | yes | one sentence, repeated | Near-zero-entropy filler. Long, but carries almost no information to overwrite the state with. |
| `natural` | yes | Wikipedia text | Ordinary prose at full entropy — the realistic case. |
| `interference` | yes | *adjacent-slot* claims: "My surname is Connor.", "My friend's name is Marcus." | Competing bindings that are similar to the fact but answer a **different** question. Does the model confuse them for the answer? |

Each answer is scored three ways: **correct** (the real name), **distractor** (a name that was
planted in the noise — recalled confidently, but wrong), or **other**.

## How to read the results

Read the `control` curve **first**. It should sit at or near zero at every budget; if it does not,
the target names are guessable from context and the other three curves mean less than they appear
to. Then compare `repeat` (should hold longest), `natural`, and `interference` — and for
`interference`, watch the *distractor* fraction, which separates "forgot it" from "confidently
answered with the wrong name".

**Runtime:** about 25-30 minutes on a Colab T4, 12-15 on an L4 or A100. Peak VRAM stays near 3-4 GB.

## 1. Environment

This notebook is **Colab-only** by design — it assumes `/content`, Colab's preinstalled CUDA
toolchain, and shell access for `wget`. There is no local/Windows code path to keep in sync.

A T4 (free tier) is enough. The `rwkv` package runs this checkpoint in `fp16`, so no Ampere-only
features are required.

In [ ]:
import sys
import torch

assert "google.colab" in sys.modules, (
    "This notebook is written for Google Colab only.\n"
    "Open it at https://colab.research.google.com and pick a GPU runtime."
)
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4 is fine)."
)

p = torch.cuda.get_device_properties(0)
print(f"GPU                : {p.name}")
print(f"VRAM               : {p.total_memory / 1e9:.1f} GB")
print(f"Compute capability : {p.major}.{p.minor}")
print(f"torch              : {torch.__version__}")
print("\nfp16 inference needs no special capability -- a T4 (7.5) and up all work.")

In [ ]:
# rwkv  -> the inference package (model + World tokenizer)
# ninja -> builds rwkv's fused CUDA kernel; without it the pure-PyTorch path is several times slower
!pip install -q rwkv ninja datasets

# 2.84 GB. -nc means a re-run of this cell is a no-op rather than a second download.
!mkdir -p models
!wget -nc -P models https://huggingface.co/BlinkDL/rwkv-7-world/resolve/main/RWKV-x070-World-1.5B-v3-20250127-ctx4096.pth

## 2. Load RWKV-7 World 1.5B

`RWKV_V7_ON` **must be set before `rwkv.model` is imported**. The package chooses between its
v4/v5/v6 and its v7 (`RWKV_x070`) implementations at import time by reading this variable. Load an
x070 checkpoint without it and you get no error — just a model running the wrong architecture and
producing quiet nonsense.

The smoke test below also prints the state layout, because the rest of the notebook depends on it:
**3 tensors per layer**, and the middle one (`state[3i+1]`, the WKV matrix that actually carries
information across time) is allocated `float32` regardless of the `fp16` strategy. That matters for
the credibility of the result — any decay measured here is the model forgetting, not fp16 rounding
error piling up over 34,000 tokens.

In [ ]:
import os

os.environ["RWKV_V7_ON"]   = "1"   # MUST precede `from rwkv.model import RWKV` -- see above
os.environ["RWKV_JIT_ON"]  = "1"
os.environ["RWKV_CUDA_ON"] = "1"   # fused CUDA kernel, built on first import via ninja
os.environ["CUDA_HOME"]    = "/usr/local/cuda"
os.environ["PATH"]         = "/usr/local/cuda/bin:" + os.environ["PATH"]

try:
    from rwkv.model import RWKV
except Exception as e:
    print(f"CUDA kernel build failed: {e}\n")
    print("Fix: set os.environ['RWKV_CUDA_ON'] = '0' in this cell, restart the runtime, rerun.")
    print("That uses the pure-PyTorch kernel -- correct, just slower.")
    raise

from rwkv.utils import PIPELINE

MODEL_PATH = "models/RWKV-x070-World-1.5B-v3-20250127-ctx4096"   # no .pth suffix: rwkv adds it

model    = RWKV(model=MODEL_PATH, strategy="cuda fp16")
pipeline = PIPELINE(model, "rwkv_vocab_v20230424")   # World models use this vocab, not 20B_tokenizer.json

out, state = model.forward(pipeline.encode("The capital of France is"), None)
print(f"\nsmoke test            : 'The capital of France is' -> {pipeline.decode([int(out.argmax())])!r}")
print(f"state tensors         : {len(state)}  ({len(state) // 3} layers x 3)")
print(f"WKV state [1]         : {tuple(state[1].shape)}  dtype={state[1].dtype}")
print(f"total state size      : {sum(t.numel() * t.element_size() for t in state) / 1e6:.1f} MB "
      f"(constant -- it does not grow with context)")

## 3. Configuration

Every knob in one place.

`BUDGETS_WORDS` is spaced densely at the low end and sparsely at the high end, because if recall
breaks it is far more likely to break in the first few thousand words than between 20k and 25k.

`CHUNK_TOKENS` is a **memory** guard, not an approximation. `RWKV.forward()` has no internal
chunking — `seq_mode = len(tokens) > 1` and then the entire list goes through the sequence kernels
in one shot, materialising activations for every position in all 24 layers at once. Handing it
34,000 tokens would be a large and needless allocation. Feeding in 1,024-token pieces and passing
the state along is **exactly equivalent**, and section 5 proves it rather than assuming it.

Set `SMOKE = True` for a ~2 minute plumbing check (3 items, budgets capped at 1,000 words) before
committing to the full run.

In [ ]:
import math
import random
import re

SEED    = 0
SMOKE   = False        # True -> tiny fast pass to check the plumbing end to end

N_ITEMS = 3 if SMOKE else 20
# Distinct (fact, question) items per condition. This is the sample size behind every recall
# number: with 20, a Wilson 95% interval on a single cell is roughly +/- 20 percentage points.

BUDGETS_WORDS = ([0, 250, 1000] if SMOKE else
                 [0, 100, 250, 500, 1000, 2000, 4000, 6000, 9000, 12000, 16000, 20000, 25000])
MAX_NOISE_WORDS = BUDGETS_WORDS[-1]

CONDITIONS = ["control", "repeat", "natural", "interference"]

CHUNK_TOKENS   = 1024  # memory guard for feed(); exactness is verified in section 5
MAX_NEW_TOKENS = 16    # answers are a name -- generation also stops at the first newline

OUT_DIR       = "/content/rnn_recall_results"
SAVE_TO_DRIVE = False                                   # section 12 copies results to Drive if True
DRIVE_DIR     = "/content/drive/MyDrive/rnn_recall_results"

random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

n_probes = len(CONDITIONS) * N_ITEMS * len(BUDGETS_WORDS)
print(f"conditions      : {CONDITIONS}")
print(f"items/condition : {N_ITEMS}")
print(f"noise budgets   : {BUDGETS_WORDS}")
print(f"total probes    : {len(CONDITIONS)} x {N_ITEMS} x {len(BUDGETS_WORDS)} = {n_probes}")
print(f"noise fed       : {len(CONDITIONS) * N_ITEMS * MAX_NOISE_WORDS:,} words "
      f"(one pass per condition-item -- see section 10)")
print(f"output dir      : {OUT_DIR}")

## 4. The two primitives

Everything below is built from exactly two functions.

**`feed(text, state)`** pushes text through the model and returns the updated state. It splits the
token list into `CHUNK_TOKENS`-sized pieces and threads the state through them, which bounds memory
without changing the result.

**`answer(state, question)`** appends the question and greedy-decodes a short reply. It works on a
**clone** of the state, so probing recall at one budget cannot contaminate the state we then carry
forward to the next budget. That is what makes the single-pass sweep in section 10 legitimate.

Decoding is pure argmax with no sampling and no repetition penalty, so nothing about the decoder
can vary between conditions — any difference in the results comes from the state.

In [ ]:
@torch.no_grad()
def feed(text, state=None):
    """Advance `state` by `text`. Returns (last_logits, new_state, n_tokens).

    state=None starts from a fresh zero state. Empty text is a no-op.
    """
    toks = pipeline.encode(text)
    if not toks:
        return None, state, 0
    out = None
    for i in range(0, len(toks), CHUNK_TOKENS):
        out, state = model.forward(toks[i:i + CHUNK_TOKENS], state)
    return out, state, len(toks)


@torch.no_grad()
def answer(state, question, max_new=MAX_NEW_TOKENS):
    """Greedy-decode one line of reply from a COPY of `state`. Never mutates the original."""
    s = [t.clone() for t in state] if state is not None else None
    out, s = model.forward(pipeline.encode(question), s)

    toks = []
    for _ in range(max_new):
        t = int(out.argmax())
        if t == 0:                       # end-of-text
            break
        toks.append(t)
        if "\n" in pipeline.decode(toks):
            break
        out, s = model.forward([t], s)

    return pipeline.decode(toks).split("\n")[0].strip()


print("feed() / answer() defined.")

## 5. Does splitting the input change the answer?

Two things later in this notebook depend on the same property, so it is worth one cell to establish
it up front:

1. `feed()` chunks long text into 1,024-token pieces (section 4).
2. The sweep feeds noise **incrementally** and probes along the way, rather than re-feeding the
   whole prefix at every budget (section 10) — a 3.2x saving on the largest budget.

Both are only valid if RWKV state-passing is associative: feeding `A` then `B` must land in the
same place as feeding `AB` in one call.

The comparison is on **argmax**, not on exact float equality. Floating-point addition is not
associative, so chunking necessarily perturbs the last bits; what matters is that the perturbation
stays at rounding scale and never changes which token wins.

In [ ]:
probe = "My name is Alderic.\n" + "The quick brown fox jumps over the lazy dog. " * 60
toks  = pipeline.encode(probe)
out_whole, _ = model.forward(toks, None)

def _relerr(a, b):
    a, b = a.float(), b.float()
    return (a - b).abs().max().item() / a.abs().max().item()

print(f"probe: {len(toks)} tokens\n")
print(f"{'split at':>9} {'rel err':>10}   argmax matches")
for k in [1, 10, 137, len(toks) // 2, len(toks) - 1]:
    _, s = model.forward(toks[:k], None)
    out_split, _ = model.forward(toks[k:], s)
    ok = int(out_whole.argmax()) == int(out_split.argmax())
    print(f"{k:>9} {_relerr(out_whole, out_split):>10.2e}   {ok}")

# And the real thing: feed()'s own CHUNK_TOKENS loop, on text long enough to exercise it.
long_probe = "The quick brown fox jumps over the lazy dog. " * 300
long_toks  = pipeline.encode(long_probe)
out_one, _       = model.forward(long_toks, None)
out_chunked, _, n = feed(long_probe)

print(f"\nfeed() chunking: {n} tokens in {math.ceil(n / CHUNK_TOKENS)} chunks of <= {CHUNK_TOKENS}")
print(f"  rel err vs one call : {_relerr(out_one, out_chunked):.2e}")
print(f"  argmax matches      : {int(out_one.argmax()) == int(out_chunked.argmax())}")

assert int(out_one.argmax()) == int(out_chunked.argmax()), (
    "feed()'s chunked path disagrees with a single forward call. Everything downstream assumes "
    "these are interchangeable -- stop here and investigate before running the sweep."
)
print("\nPASS -- chunked and incremental feeding are interchangeable with feeding it all at once.")

## 6. The facts, the question, and the distractors

20 items, each a single first-person sentence and one question about it. The names are deliberately
uncommon: if the target were "James", a model that had forgotten the fact could still score
"correct" by falling back on a common name, and the recall numbers would be inflated. The `control`
condition measures exactly how much of that is left.

The **distractor** names are a disjoint pool used only by the interference noise, so a scored answer
is never ambiguous about which pool it came from.

The interference noise asserts *adjacent* attributes — a surname, a friend's name, a dog's name —
rather than contradicting the fact outright. These are claims that look and sound like the answer
but respond to a different question, which is the harder and more realistic confusion to test.

In [ ]:
FACT_TEMPLATE = "My name is {name}.\n"
QUESTION      = "\n\nQuestion: What is my name?\nAnswer:"

TARGET_NAMES = [
    "Alderic", "Thessaly", "Corvin", "Mirabel", "Lucan",
    "Ottoline", "Ferris", "Isolde", "Ansel", "Perpetua",
    "Barnaby", "Cressida", "Wilhelmina", "Cormac", "Seraphina",
    "Aurelio", "Marguerite", "Leopold", "Rosalind", "Peregrine",
]

DISTRACTOR_NAMES = [
    "Connor", "Marcus", "Rufus", "Elliot", "Priya", "Sylvia",
    "Dermot", "Yolanda", "Hugo", "Ingrid", "Tobias", "Delia",
    "Rashid", "Freya", "Milo", "Bianca", "Owen", "Nadia",
    "Gareth", "Lorna", "Emil", "Saoirse", "Vaughn", "Clara",
]

# Adjacent slots: each states a name, none of them states MY name.
DISTRACTOR_SLOTS = [
    "My surname is {name}.",
    "My middle name is {name}.",
    "My friend's name is {name}.",
    "My brother's name is {name}.",
    "My neighbour's name is {name}.",
    "My manager's name is {name}.",
    "My dog's name is {name}.",
    "My landlord's name is {name}.",
    "My doctor's name is {name}.",
    "My cousin's name is {name}.",
]

assert len(TARGET_NAMES) >= N_ITEMS, f"need {N_ITEMS} target names, have {len(TARGET_NAMES)}"
assert not (set(TARGET_NAMES) & set(DISTRACTOR_NAMES)), "target and distractor pools must be disjoint"
assert len(set(TARGET_NAMES)) == len(TARGET_NAMES), "duplicate target name"

print("An item looks like this end to end:\n")
print("=" * 70)
print(FACT_TEMPLATE.format(name=TARGET_NAMES[0]), end="")
print("   <<< 0 to 25,000 words of noise go here >>>", end="")
print(QUESTION)
print("=" * 70)
print(f"\n{len(TARGET_NAMES)} target names, {len(DISTRACTOR_NAMES)} distractor names, "
      f"{len(DISTRACTOR_SLOTS)} adjacent slots.")

## 7. The three noise sources

Each generator returns a **word list of exactly `MAX_NOISE_WORDS`**, deterministic in
`(kind, item_idx)`. Returning the full 25,000-word stream once and slicing prefixes off it is what
makes the single-pass sweep exact: the noise at 4,000 words is, by construction, character-for-
character the first 4,000 words of the noise at 25,000.

- **repeat** — one sentence, over and over. Long but nearly information-free.
- **natural** — a distinct, non-overlapping window of WikiText-2 per item. `control` reuses the
  *same* window as `natural` for the same item, so the floor is a matched counterfactual: identical
  noise, identical question, the only difference is whether the fact was ever given.
- **interference** — adjacent-slot assertions drawn from the pools in section 6.

`noise_slices()` cuts the stream at the budget checkpoints so that concatenating the pieces
reproduces the full string exactly — asserted below, because an off-by-one space here would quietly
change what every arm is actually being fed.

In [ ]:
from datasets import load_dataset

# WikiText-2 (raw) is parquet-backed on the Hub -- no trust_remote_code, ~6 MB, a few seconds.
_ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
_lines = [t.strip() for t in _ds["text"] if t.strip() and not t.strip().startswith("=")]

# WikiText ships Moses-tokenised: punctuation is spaced off its word and hyphens/decimals are
# written " @-@ " / " @.@ ". Left alone, the "ordinary prose" arm would not actually be ordinary
# prose -- it would be prose with a token distribution no real text has. Undo it.
_raw = " ".join(_lines).replace(" @-@ ", "-").replace(" @,@ ", ",").replace(" @.@ ", ".")
_raw = re.sub(r' ([,.;:!?%\)\]])', r'\1', _raw)      # " word ."  -> " word."
_raw = re.sub(r'([\(\[]) ', r'\1', _raw)             # "( word"   -> "(word"
WIKI_WORDS = _raw.split()

need = N_ITEMS * MAX_NOISE_WORDS
print(f"WikiText-2 train : {len(WIKI_WORDS):,} words")
print(f"needed           : {N_ITEMS} items x {MAX_NOISE_WORDS:,} = {need:,} words "
      f"(non-overlapping windows)")
assert len(WIKI_WORDS) >= need, "corpus too small -- switch to wikitext-103-raw-v1"


def make_noise(kind, item_idx, n_words=MAX_NOISE_WORDS):
    """Exactly `n_words` words of noise, deterministic in (kind, item_idx)."""
    rng = random.Random(SEED * 1_000 + item_idx)

    if kind == "repeat":
        sent  = "The quick brown fox jumps over the lazy dog.".split()
        words = sent * (n_words // len(sent) + 1)

    elif kind in ("natural", "control"):
        start = item_idx * n_words          # one non-overlapping window per item
        words = WIKI_WORDS[start:start + n_words]

    elif kind == "interference":
        words = []
        while len(words) < n_words:
            words += rng.choice(DISTRACTOR_SLOTS).format(name=rng.choice(DISTRACTOR_NAMES)).split()

    else:
        raise ValueError(f"unknown noise kind {kind!r}")

    assert len(words) >= n_words, f"{kind}: produced {len(words)} words, needed {n_words}"
    return words[:n_words]


def noise_slices(words, budgets):
    """Yield (budget, text) so that concatenating every text reproduces ' '.join(words) exactly.

    The sweep feeds these in order, probing recall after each one, so slice k must be precisely
    the stretch of noise between budget k-1 and budget k.
    """
    prev = 0
    for b in budgets:
        chunk = words[prev:b]
        text  = ("" if not chunk else (" " if prev else "") + " ".join(chunk))
        yield b, text
        prev = b


# The slicing has to be exact, so check it rather than trust it.
for kind in ("repeat", "natural", "interference"):
    w = make_noise(kind, 0)
    assert len(w) == MAX_NOISE_WORDS
    rebuilt = "".join(text for _, text in noise_slices(w, BUDGETS_WORDS))
    assert rebuilt == " ".join(w), f"{kind}: budget slices do not reconstruct the noise stream"
    print(f"{kind:>13} : {len(w):,} words -> slices reconstruct the stream exactly")

print("\nsample of each source (first 180 characters)\n" + "-" * 78)
for kind in ("repeat", "natural", "interference"):
    print(f"[{kind}]\n{' '.join(make_noise(kind, 0)[:40])[:180]}...\n")

## 8. Scoring

Three outcomes per probe:

- **correct** — the target name appears in the reply.
- **distractor** — no target, but a name that was planted in the noise does appear. The model
  answered confidently with the wrong thing rather than losing the thread entirely.
- **other** — neither. Forgot it, refused, or wandered off.

Matching is on whole words after lowercasing and stripping punctuation, so `"Alderic."`,
`"alderic"` and `"Your name is Alderic"` all count, while a chance substring cannot. If a reply
somehow contains both a target and a distractor, **correct** wins — it did produce the right answer.

In [ ]:
_WORD_RE = re.compile(r"[a-z0-9']+")

def _wordset(s):
    return set(_WORD_RE.findall(s.lower()))


def score(response, target, distractors=DISTRACTOR_NAMES):
    """-> 'correct' | 'distractor' | 'other'"""
    w = _wordset(response)
    if target.lower() in w:
        return "correct"
    if w & {d.lower() for d in distractors}:
        return "distractor"
    return "other"


_TESTS = [
    ("Alderic",                          "Alderic", "correct"),
    ("Your name is Alderic.",            "Alderic", "correct"),
    ("alderic!",                         "Alderic", "correct"),
    ("I think it is Connor.",            "Alderic", "distractor"),
    ("Your name is Alderic, not Connor", "Alderic", "correct"),     # target wins over distractor
    ("I do not know.",                   "Alderic", "other"),
    ("",                                 "Alderic", "other"),
    ("Aldericus",                        "Alderic", "other"),       # whole words only
]
for text, target, expected in _TESTS:
    got = score(text, target)
    assert got == expected, f"score({text!r}, {target!r}) = {got!r}, expected {expected!r}"
    print(f"  {expected:>10}  <-  {text!r}")

print(f"\nscoring self-test: {len(_TESTS)}/{len(_TESTS)} passed.")

## 9. One probe, end to end

Before spending half an hour on 1,040 probes, run one item by hand and look at the actual text.

Three cases, at 0 words of noise and again at 1,000:

- **with the fact** — should answer with the target name,
- **without the fact** (the control) — should *not*; if it does, the item is guessable,
- **with the fact, after interference noise** — the interesting case.

If the "without the fact" row ever returns the target name, stop and change the name pool: the
sweep would be measuring the model's priors, not its memory.

In [ ]:
demo_name = TARGET_NAMES[0]
demo_fact = FACT_TEMPLATE.format(name=demo_name)

print("EXACT text fed to the model (0-noise case)")
print("-" * 78)
print(repr(demo_fact + QUESTION))
print("-" * 78)

for budget in (0, 1000):
    print(f"\n=== {budget} words of noise ===")
    for label, kind, with_fact in [("with fact      ", "natural",      True),
                                   ("no fact (ctrl) ", "natural",      False),
                                   ("with fact + int", "interference", True)]:
        state = feed(demo_fact)[1] if with_fact else None
        if budget:
            noise = " ".join(make_noise(kind, 0, budget))
            _, state, n_tok = feed(noise, state)
        else:
            n_tok = 0
        resp = answer(state, QUESTION)
        print(f"  {label} [{kind:>12}, {n_tok:>5} noise tokens] "
              f"-> {resp[:44]!r:<48} {score(resp, demo_name)}")

## 10. The sweep

The whole experiment, in one loop.

The thing worth understanding here is why it is affordable. The naive version re-feeds the whole
noise prefix at every budget: 0 + 100 + 250 + ... + 25,000 = **79,350 words per condition-item**.
But budget *k* is a strict prefix of budget *k+1*, and section 5 established that state-passing is
associative — so we can feed the stream **once**, pausing at each checkpoint to probe from a clone
of the state. That is **25,000 words per condition-item**, a 3.2x saving, and it is exactly
equivalent rather than an approximation.

`answer()` cloning the state is what makes this safe: the question tokens are never written into
the state that continues into the next budget.

Progress prints a pooled recall per condition as it finishes. Expect roughly 25-30 minutes on a T4.

In [ ]:
import time
import pandas as pd
from tqdm.auto import tqdm

torch.cuda.reset_peak_memory_stats()
t0, rows = time.time(), []
bar = tqdm(total=len(CONDITIONS) * N_ITEMS, desc="condition x item passes")

for cond in CONDITIONS:
    noise_kind   = "natural" if cond == "control" else cond   # control = natural noise, fact withheld
    fact_present = cond != "control"

    for item in range(N_ITEMS):
        target = TARGET_NAMES[item]

        # 1. Seed the state with the fact -- or leave it empty, which is the whole point of the control.
        state = feed(FACT_TEMPLATE.format(name=target))[1] if fact_present else None

        # 2. Walk the noise stream once, probing from a clone at every budget checkpoint.
        words, cum_tokens = make_noise(noise_kind, item), 0
        for budget, text in noise_slices(words, BUDGETS_WORDS):
            if text:
                _, state, n = feed(text, state)
                cum_tokens += n
            resp = answer(state, QUESTION)
            rows.append(dict(condition=cond, item=item, target=target,
                             budget_words=budget, noise_tokens=cum_tokens,
                             outcome=score(resp, target), response=resp[:60]))
        bar.update(1)

    seen = [r for r in rows if r["condition"] == cond]
    hits = sum(r["outcome"] == "correct" for r in seen)
    bar.write(f"{cond:>13} done -- recall pooled over every budget: {hits / len(seen):.1%}")

bar.close()
results = pd.DataFrame(rows)

expected = len(CONDITIONS) * N_ITEMS * len(BUDGETS_WORDS)
assert len(results) == expected, f"expected {expected} probes, got {len(results)}"

print(f"\n{len(results)} probes in {(time.time() - t0) / 60:.1f} min")
print(f"noise fed        : {results.groupby(['condition', 'item']).noise_tokens.max().sum():,} tokens")
print(f"peak GPU memory  : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## 11. The numbers

Recall rate is a proportion out of `N_ITEMS`, so intervals use the **Wilson score interval** rather
than the textbook normal approximation. The difference matters precisely where this experiment
spends most of its time: at 20/20 and 0/20, the normal approximation gives a zero-width interval,
which is obviously wrong — Wilson does not.

The last table is the concrete headline: the noise budget at which each condition's recall curve
first falls **below 90%** and **below 50%**, linearly interpolated between the two budgets that
straddle the crossing.

It is the *first* crossing, so read it next to figure 1 rather than on its own — with 20 items a
curve can dip below a threshold on sampling noise and recover, and this table will report the dip.
`never` means it was still above that level at 25,000 words; `n/a` means it was never above it at
all, which is what the `control` row should say.

In [ ]:
def wilson(k, n, z=1.96):
    """Wilson score interval for k successes in n trials. Well behaved at k=0 and k=n."""
    if n == 0:
        return float("nan"), float("nan")
    p, denom = k / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half   = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


summary = (results.assign(hit=results.outcome.eq("correct"))
           .groupby(["condition", "budget_words"], as_index=False)
           .agg(n=("hit", "size"), k=("hit", "sum"), noise_tokens=("noise_tokens", "mean")))
summary["recall"] = summary.k / summary.n
summary = pd.concat(
    [summary, pd.DataFrame([wilson(k, n) for k, n in zip(summary.k, summary.n)],
                           columns=["ci_lo", "ci_hi"], index=summary.index)], axis=1)

print("RECALL RATE  (rows = noise words, columns = condition)")
print("-" * 62)
print(summary.pivot(index="budget_words", columns="condition", values="recall")[CONDITIONS]
             .round(3).to_string())

print("\n\nOUTCOME MIX, pooled over all budgets")
print("-" * 62)
comp = (results.groupby(["condition", "outcome"]).size().unstack(fill_value=0)
        .reindex(index=CONDITIONS, columns=["correct", "distractor", "other"], fill_value=0))
print(comp.div(comp.sum(axis=1), axis=0).round(3).to_string())


def crossing(g, level):
    """First word budget where recall drops below `level`, interpolated. NaN = never, in range."""
    g = g.sort_values("budget_words")
    x, y = list(g.budget_words), list(g.recall)
    if y[0] < level:
        return 0.0
    for i in range(1, len(x)):
        if y[i] < level <= y[i - 1]:
            return x[i - 1] + (y[i - 1] - level) / (y[i - 1] - y[i]) * (x[i] - x[i - 1])
    return float("nan")


def _fmt(v):
    if math.isnan(v):
        return "never"       # still above the level at the largest budget swept
    if v == 0.0:
        return "n/a"         # never above it in the first place -- expected for `control`
    return f"{v:,.0f}"


print("\n\nWHERE IT BREAKS")
print("-" * 80)
print(f"{'condition':>13} {'recall @0':>11} {'recall @max (95% CI)':>22} "
      f"{'words to <90%':>14} {'words to <50%':>14}")
for cond in CONDITIONS:
    g  = summary[summary.condition == cond]
    r0 = g.loc[g.budget_words == 0, "recall"].iloc[0]
    rm = g.loc[g.budget_words == MAX_NOISE_WORDS, "recall"].iloc[0]
    lo, hi = wilson(*g.loc[g.budget_words == MAX_NOISE_WORDS, ["k", "n"]].iloc[0])
    print(f"{cond:>13} {r0:>11.1%} {f'{rm:.1%} [{lo:.0%}, {hi:.0%}]':>22} "
          f"{_fmt(crossing(g, 0.9)):>14} {_fmt(crossing(g, 0.5)):>14}")

print(f"\nn = {N_ITEMS} items per cell.  'never' = still above that level at "
      f"{MAX_NOISE_WORDS:,} words;  'n/a' = never above it even at zero noise.")

## 12. Graphs

Three figures, saved to `OUT_DIR` alongside the raw per-probe CSV.

**Figure 1** is the headline, plotted twice. The left panel uses **words**, which is how the
experiment was specified. The right uses **tokens**, and it is the one to trust when comparing the
arms against each other: the three noise sources have different tokens-per-word ratios (the
repeated sentence is cheap, the interference sentences are not), so equal words is *not* equal
work through the state. The x-axis is symlog so that the 0-noise point is on the same plot as
25,000 without the low end being squashed.

**Figure 2** breaks each fact-bearing condition into its three outcomes. A rising grey band means
the fact is being forgotten; a rising orange band in the `interference` panel means something more
specific — the model is answering with a name it read in the noise.

**Figure 3** drops down to individual items, to show whether a collapse is uniform across all 20
facts or driven by a handful of unlucky names.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

COLOR   = {"control": "#8a8a8a", "repeat": "#1b9e77", "natural": "#7570b3", "interference": "#d95f02"}
OUTCOME = {"correct": "#1b9e77", "distractor": "#d95f02", "other": "#c8c8c8"}
ARMS    = [c for c in CONDITIONS if c != "control"]

# --- Figure 1: recall vs noise, on both axes -------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
for ax, xcol, xlabel, title in [
        (axes[0], "budget_words", "noise (words)",       "Recall vs noise in words"),
        (axes[1], "noise_tokens", "noise (RWKV tokens)", "Recall vs noise in tokens (fair across arms)")]:
    for cond in CONDITIONS:
        g = summary[summary.condition == cond].sort_values("budget_words")
        ax.fill_between(g[xcol], g.ci_lo, g.ci_hi, color=COLOR[cond], alpha=0.15, lw=0)
        ax.plot(g[xcol], g.recall, marker="o", ms=4, color=COLOR[cond], label=cond,
                ls="--" if cond == "control" else "-", lw=1.6 if cond == "control" else 2.0)
    ax.axhline(0.5, color="k", lw=0.7, ls=":")
    # symlog (not log) so the 0-noise point is on the same axis as 25,000. It is symmetric about
    # zero by default, so the limits must be pinned or a third of the panel is negative decades.
    ax.set_xscale("symlog", linthresh=250, linscale=0.6)
    ax.set_xlim(-40, summary[xcol].max() * 1.25)
    ax.set(xlabel=xlabel, ylabel="recall rate", ylim=(-0.05, 1.05), title=title)
    ax.grid(alpha=0.3)

# One shared legend under both panels -- inside either one it would sit on top of a curve.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(CONDITIONS), loc="lower center",
           bbox_to_anchor=(0.5, -0.07), frameon=False, fontsize=10)
fig.suptitle(f"Does the fact survive the noise?  RWKV-7 World 1.5B, n={N_ITEMS} items per point, "
             f"bands = Wilson 95% CI", y=1.02, fontsize=11)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/fig1_recall.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 2: what it said when it was not correct ------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharey=True)
for ax, cond in zip(axes, ARMS):
    tab = (results[results.condition == cond]
           .groupby(["budget_words", "outcome"]).size().unstack(fill_value=0)
           .reindex(index=BUDGETS_WORDS, columns=list(OUTCOME), fill_value=0))
    tab = tab.div(tab.sum(axis=1), axis=0)
    x, bottom = np.arange(len(tab)), np.zeros(len(tab))
    ax.grid(axis="y", alpha=0.3, zorder=0)
    for outcome, colour in OUTCOME.items():
        ax.bar(x, tab[outcome].values, bottom=bottom, color=colour, width=0.82,
               label=outcome, zorder=3, edgecolor="white", linewidth=0.5)
        bottom += tab[outcome].values
    ax.set_xticks(x)
    ax.set_xticklabels([f"{b:,}" for b in tab.index], rotation=45, ha="right", fontsize=10)
    ax.tick_params(axis="y", labelsize=10)
    ax.set(title=cond, xlabel="noise (words)", ylim=(0, 1))
    ax.title.set_fontsize(13)
    ax.xaxis.label.set_fontsize(11)
axes[0].set_ylabel("fraction of probes", fontsize=11)
axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=11)  # outside the bars
fig.suptitle("Outcome mix per condition -- grey is forgetting, orange is answering with a "
             "name from the noise", y=1.03, fontsize=13)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/fig2_outcomes.png", dpi=160, bbox_inches="tight")
plt.show()

# --- Figure 3: per item, is the collapse uniform? --------------------------------------
LEVEL = {"other": 0, "distractor": 1, "correct": 2}
cmap  = ListedColormap([OUTCOME["other"], OUTCOME["distractor"], OUTCOME["correct"]])

fig, axes = plt.subplots(1, 3, figsize=(17, 7.5), sharey=True)
for ax, cond in zip(axes, ARMS):
    sub = results[results.condition == cond]
    mat = (sub.assign(v=sub.outcome.map(LEVEL))
           .pivot(index="item", columns="budget_words", values="v")
           .reindex(columns=BUDGETS_WORDS).values)
    ax.imshow(mat, aspect="auto", cmap=cmap, vmin=-0.5, vmax=2.5, interpolation="nearest")
    # White gridlines between cells -- without them, adjacent same-colour cells fuse into one
    # blob and it is impossible to see the grid or count budgets/items at a glance.
    ax.set_xticks(np.arange(-0.5, len(BUDGETS_WORDS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, N_ITEMS, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", length=0)
    ax.set_xticks(range(len(BUDGETS_WORDS)))
    ax.set_xticklabels([f"{b:,}" for b in BUDGETS_WORDS], rotation=45, ha="right", fontsize=10)
    ax.set(title=cond, xlabel="noise (words)")
    ax.title.set_fontsize(13)
    ax.xaxis.label.set_fontsize(11)
axes[0].set_yticks(range(N_ITEMS))
axes[0].set_yticklabels(TARGET_NAMES[:N_ITEMS], fontsize=10)
axes[0].set_ylabel("fact item", fontsize=11)
axes[-1].legend(handles=[Patch(facecolor=c, label=k) for k, c in OUTCOME.items()],
                loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=11)
fig.suptitle("Every probe, one cell each -- is the collapse uniform or name-specific?",
             y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/fig3_per_item.png", dpi=160, bbox_inches="tight")
plt.show()

# --- Save the raw data -----------------------------------------------------------------
results.to_csv(f"{OUT_DIR}/results.csv", index=False)
summary.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"written to {OUT_DIR}:")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:<22} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1024:>8.1f} KB")

if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    shutil.copytree(OUT_DIR, DRIVE_DIR, dirs_exist_ok=True)
    print(f"\ncopied to {DRIVE_DIR}")
else:
    print("\n/content is wiped when the runtime ends. Set SAVE_TO_DRIVE = True in section 3 to keep these.")

## 13. Reading the result

Work through it in this order.

**1. Is the floor actually a floor?** Look at `control` in figure 1. It should be flat and near
zero. It is a matched counterfactual — identical Wikipedia noise, identical question, the fact
simply never given — so whatever it sits at is the score a model gets for free. Subtract it
mentally from every other curve. If it drifts *upward* with more noise, the model is picking names
out of the Wikipedia text and guessing; if that happens, the honest fix is a more distinctive name
pool, not a different metric.

**2. Where does each arm cross?** The "words to <50%" column in section 11 is the number to quote.
The expected ordering is `repeat` > `natural` > `interference`: a repeated sentence writes almost
no new information into the state, ordinary prose writes a lot, and interference writes things
shaped like the answer. If `repeat` holds all the way to 25,000 words while `natural` collapses at
a few thousand, the limit is **interference in the state**, not elapsed time or token count.

**3. Words or tokens?** Compare the two panels of figure 1. If the arms separate on the word axis
but line up on the token axis, the differences were mostly a tokenisation artifact and the real
story is simply "N tokens of anything". If they stay separated on the token axis, the *content* of
the noise genuinely matters.

**4. Forgetting or overwriting?** Figure 2, `interference` panel. Grey growing means the fact
faded. Orange growing means something sharper: the model still believes it knows the answer and
produces a name it read in the noise. Those are different failure modes and only the second one is
really "interference".

**5. Uniform or name-specific?** Figure 3. Horizontal stripes mean particular names are fragile —
probably a tokenisation quirk in that name — and the aggregate curve is an average over a bimodal
population. A clean vertical gradient means the collapse is a genuine property of the budget.

### If you want to push further

- **Tighter intervals.** Everything here is `n = 20`, so a single cell is +/- ~20 pp. Raising
  `N_ITEMS` to 40 halves nothing but does narrow it to ~15 pp; runtime scales linearly.
- **Where the fact sits.** It is at position 0 in every probe. Putting it in the *middle* of the
  noise turns this into a recency test, which is a different and also interesting question.
- **State capacity.** RWKV-7 World ships at 0.1B / 0.4B / 1.5B / 2.9B, and the recurrent state
  scales with width and depth. Swapping `MODEL_PATH` in section 2 and re-running gives a
  state-size-versus-retention curve.
- **Bigger budgets.** Nothing in this notebook caps out at 25,000 words. `BUDGETS_WORDS` can go
  further; the cost is linear and memory is flat, which is the whole advantage of a recurrent state.